# Fast tomo Procedure in Python 

In [ ]:
# imports
import tomobase
import os
import ncempy
import stackview
import numpy as np
import pandas as pd 
import logging
tomobase.logger.setLevel(logging.WARNING)
directory = r'\\ematbyname\emat\TimC\USC'
subdirectory = 'B10'
ser_file = 'ftomo2_1.ser'
csv_file = 'angle_log_B10-3.csv'

imgs = []
for i in range(10000):
    try:
        imgs.append(ncempy.io.ser.fileSER(os.path.join(directory, subdirectory, ser_file)).getDataset(i)[0])
    except:
        break
    

data = np.stack(imgs, axis=0)
#stackview.slice(data)

df = pd.read_csv(os.path.join(directory, subdirectory, csv_file))
df.columns = df.columns.str.strip()
df = df.loc[:, ~df.columns.str.contains(r'^Unnamed')]

arr = df.iloc[:, :2].astype(float).to_numpy()
shape = data.shape[0]


t = np.linspace(arr[0, 0], arr[-1, 0], shape)

angles = np.zeros(shape, dtype=float)

angles[0] = arr[0, 1]
angles[-1] = arr[-1, 1]
for i in range(0, arr.shape[0]-1):
    mask = (t >= arr[i, 0]) & (t < arr[i+1, 0])
    angles[mask] = arr[i, 1]
    


sino = tomobase.data.Sinogram(data, angles)


In [ ]:
#sino = tomobase.processes.align_sinogram_center_of_mass(sino)
#sino = tomobase.processes.bin(sino)
sino = tomobase.processes.background_subtract_median(sino)
sino = tomobase.processes.align_sinogram_xcorr(sino)
#sino = tomobase.processes.align_tilt_axis_shift(sino)


In [ ]:
# translate x and y 
# import circle shift
from scipy.ndimage import center_of_mass, shift, rotate
translate_x = 0
translate_y = -100
sino.data = shift(sino.data, (0, translate_y, translate_x), mode='wrap')
stackview.slice(sino.data)


In [40]:
stackview.crop(sino.data)

_Cropper(children=(HBox(children=(VBox(children=(VBox(children=(IntRangeSlider(value=(0, 392), description='Z'…

In [41]:
import copy
sino_cropped = copy.deepcopy(sino)
cropx = slice(63,310)
cropy = slice(0,260)
sino_cropped.data = sino_cropped.data[:, cropx, cropy]

sino_cropped = tomobase.processes.align_sinogram_xcorr(sino_cropped)
stackview.slice(sino_cropped.data)

#sino = tomobase.processes.align_tilt_axis_shift(sino)


100%|██████████| 392/392 [00:00<00:00, 48827.46it/s]


In [42]:
sino = sino_cropped

In [21]:
sino = tomobase.processes.align_sinogram_xcorr(sino)
sino = tomobase.processes.align_tilt_axis_shift(sino)


100%|██████████| 336/336 [00:04<00:00, 80.01it/s]

100%|██████████| 336/336 [00:04<00:00, 83.70it/s]

100%|██████████| 336/336 [00:04<00:00, 83.42it/s]

100%|██████████| 21/21 [05:55<00:00, 16.95s/it]


In [23]:
import copy
import numpy as np
from scipy.signal.windows import hann
from skimage.registration import phase_cross_correlation


def get_bad_images_origin(sino, recon_iters=1, upsample=10, poly_order=2, thresh=1.5):
    s = copy.deepcopy(sino)
    # normalize intensity per-projection (optional)
    s_min = s.data.min()
    s_max = s.data.max()
    s.data = (s.data - s_min) / (s_max)
    # reconstruct and forward-project (use same functions you used before)
    rec = tomobase.processes.optomo_reconstruct(s, iterations=recon_iters)
    sino_sim = tomobase.processes.project(rec, s.angles)
    for i in range(sino_sim.data.shape[0]):
        sino_sim.data[i, :, :] = sino_sim.data[i, :, :] / np.sum(sino_sim.data[i, :, :])
    sino_sim.data = (sino_sim.data - sino_sim.data.min()) / (sino_sim.data.max())
    #sino_sim.data = (sino_sim.data - sino_sim.data.min()) / (sino_sim.data.max() - sino_sim.data.min())
    # optional: equalize intensity per-projection if you have that function
    # sino_sim = hv_EqualizeIntensity(sino_sim)
    n = s.data.shape[0]
    diff = np.zeros_like(s.angles, dtype=float)
    for i in range(n):
        ref = sino_sim.data[i, :, :]
        mov = s.data[i, :, :]
        # compute subpixel shift; returns (shift_y, shift_x), error, phasediff
        shift, error, phasediff = phase_cross_correlation(ref, mov, upsample_factor=upsample)
        diff[i] = float(np.abs(error))
    # detrend and score as in your code
    coeffs = np.polyfit(s.angles, diff, poly_order)
    trend = np.polyval(coeffs, s.angles)
    residuals = diff - trend
    residuals -= residuals.min()
    median = np.median(residuals)
    MAD = np.median(np.abs(residuals - median))
    if MAD == 0:
        MAD = 1e-6
    score = (0.675 * (residuals - median)) / MAD
    idx = np.where(score > thresh)[0]  # zero-based indices
    return sino_sim, idx, score


def _prep_slice(img):
    x = np.asarray(img, dtype=np.float32)
    if np.isnan(x).any():
        x = np.nan_to_num(x, nan=float(np.nanmean(x)))
    m = float(np.mean(x)); s = float(np.std(x))
    x = (x - m) / s if s > 0 else x*0.0
    return x

def _hann2d(h, w):
    wy = hann(h, sym=False).astype(np.float32)
    wx = hann(w, sym=False).astype(np.float32)
    return wy[:, None] * wx[None, :]

def _rolling_median(y, win):
    # 1D rolling median with edge reflection
    y = np.asarray(y, float)
    win = max(3, int(win) | 1)  # odd >=3
    pad = win // 2
    yp = np.pad(y, pad, mode='reflect')
    out = np.empty_like(y)
    for i in range(len(y)):
        out[i] = np.median(yp[i:i+win])
    return out

def get_bad_images(sino, recon_iters=20, upsample=30, poly_order=None, thresh=3,
                   use_mean_std=False, combine_shift=False, overlap_ratio=0.1, crop_center=None):
    """
    Returns:
      idx: indices of likely-bad projections (0-based)
      score: robust z-like score (higher = worse)
      diff: raw metric before baseline removal (higher = worse)
    """
    s = copy.deepcopy(sino)
    s.data = (s.data-np.min(s.data))/(np.max(s.data)-np.min(s.data))
    # Reconstruct & forward-project
    rec = tomobase.processes.optomo_reconstruct(s, iterations=recon_iters)
    sino_sim = tomobase.processes.project(rec, s.angles)

    n, H, W = s.data.shape
    assert sino_sim.data.shape == s.data.shape, \
        f"Shape mismatch: {sino_sim.data.shape} vs {s.data.shape}"

    win2d = _hann2d(H, W)
    diff_error = np.zeros(n, dtype=float)
    diff_shift = np.zeros(n, dtype=float)

    for i in range(n):
        ref = _prep_slice(sino_sim.data[i])
        mov = _prep_slice(s.data[i])

        if crop_center:
            cy, cx = H//2, W//2
            hy, hx = int(H*crop_center/2), int(W*crop_center/2)
            ys, ye = cy-hy, cy+hy
            xs, xe = cx-hx, cx+hx
            ref = ref[ys:ye, xs:xe]
            mov = mov[ys:ye, xs:xe]
            win = _hann2d(ref.shape[0], ref.shape[1])
        else:
            win = win2d

        ref *= win
        mov *= win

        shift, error, _ = phase_cross_correlation(
            ref, mov,
            upsample_factor=upsample,
            overlap_ratio=overlap_ratio
        )
        diff_error[i] = float(error)                     # 0 (good) → 1 (bad)
        diff_shift[i] = float(np.hypot(shift[0], shift[1]))  # pixels, >=0

    # Pick metric: PCC error is intensity-invariant like your MATLAB p(1).
    diff = diff_error.copy()
    if combine_shift:
        # Blend in shift magnitude to help when error saturates
        # normalize to comparable scale
        s_norm = diff_shift / (np.median(diff_shift) + 1e-6)
        diff = 0.7*diff_error + 0.3*np.tanh(0.3*s_norm)

    # ---- Baseline removal (robust) ----
    # Use rolling median instead of high-order poly (prevents overfitting spikes).
    # Window ~5–11 projections often works well; tune if your angles are dense.
    baseline = _rolling_median(diff, win=11)
    residuals = diff - baseline

    # ---- Robust scoring ----
    if use_mean_std:
        mu = float(np.mean(residuals))
        sd = float(np.std(residuals)) or 1e-6
        score = (residuals - mu) / sd
    else:
        med = float(np.median(residuals))
        MAD = float(np.median(np.abs(residuals - med))) or 1e-6
        # 0.675 ~ 1/1.4826 to map MAD to sigma
        score = (0.675 * (residuals - med)) / MAD

    idx = np.where(score > thresh)[0]
    return sino_sim, idx, score


In [ ]:
# rotate sinogram 90 degrees
sino.data = np.transpose(sino.data, (0,2,1))
stackview.slice(sino.data)

In [34]:

#sino = tomobase.processes.align_tilt_axis_shift(sino)
#sino = tomobase.processes.align_tilt_axis_rotation(sino)
sino_test, idx, score = get_bad_images_origin(sino, recon_iters=1, poly_order=12)
print(idx)


stackview.curtain(sino_test.data, sino.data)
#stackview.curtain(sino_test.data, sino.data)
#stackview.slice( sino.data)

100%|██████████| 1/1 [00:00<00:00, 69.29it/s]

100%|██████████| 1/1 [00:00<00:00, 67.58it/s]

100%|██████████| 1/1 [00:00<00:00, 70.94it/s]

100%|██████████| 1/1 [00:00<00:00, 72.25it/s]

100%|██████████| 1/1 [00:00<00:00, 66.99it/s]

100%|██████████| 1/1 [00:00<00:00, 66.38it/s]

100%|██████████| 1/1 [00:00<00:00, 66.49it/s]

100%|██████████| 1/1 [00:00<00:00, 64.93it/s]

100%|██████████| 1/1 [00:00<00:00, 69.70it/s]

100%|██████████| 1/1 [00:00<00:00, 68.87it/s]

100%|██████████| 1/1 [00:00<00:00, 69.73it/s]

100%|██████████| 1/1 [00:00<00:00, 71.91it/s]

100%|██████████| 1/1 [00:00<00:00, 73.97it/s]

100%|██████████| 1/1 [00:00<00:00, 68.45it/s]

100%|██████████| 1/1 [00:00<00:00, 68.94it/s]

100%|██████████| 1/1 [00:00<00:00, 69.74it/s]

100%|██████████| 1/1 [00:00<00:00, 68.70it/s]

100%|██████████| 1/1 [00:00<00:00, 84.24it/s]

100%|██████████| 1/1 [00:00<00:00, 82.30it/s]

100%|██████████| 1/1 [00:00<00:00, 81.53it/s]

100%|██████████| 1/1 [00:00<00:00, 72.96it/s]

100%|████████

[ 83  84  85  86  87  88  89 106 108 110 181 183 187 200 203 213 214 216]


In [ ]:

sino = tomobase.processes.align_tilt_axis_shift(sino)
sino = tomobase.processes.align_tilt_axis_rotation(sino)

In [43]:
idx = [304]
sino.to_file(os.path.join(directory, subdirectory, 'sino_aligned.mrc'))

In [ ]:



print(idx)
print(score)
stackview.slice(sino.data)
#sino = tomobase.processes.align_sinogram_xcorr(sino)




In [44]:
sino.remove(idx)
sino = tomobase.processes.align_sinogram_xcorr(sino)
#sino = tomobase.processes.align_tilt_axis_shift(sino)

  0%|          | 0/390 [00:00<?, ?it/s]

100%|██████████| 391/391 [00:00<00:00, 64365.67it/s]


In [47]:
#new_sino=copy.deepcopy(sino)
new_sino = tomobase.processes.align_tilt_axis_shift(new_sino)
stackview.slice(new_sino.data)

  0%|          | 0/21 [00:00<?, ?it/s]


100%|██████████| 247/247 [00:01<00:00, 128.66it/s]

100%|██████████| 247/247 [00:01<00:00, 130.58it/s]

100%|██████████| 247/247 [00:01<00:00, 134.70it/s]

100%|██████████| 247/247 [00:01<00:00, 130.07it/s]

100%|██████████| 247/247 [00:01<00:00, 133.16it/s]

100%|██████████| 247/247 [00:01<00:00, 134.23it/s]

100%|██████████| 247/247 [00:01<00:00, 131.00it/s]

100%|██████████| 247/247 [00:01<00:00, 129.82it/s]

100%|██████████| 247/247 [00:01<00:00, 128.41it/s]

100%|██████████| 247/247 [00:01<00:00, 131.31it/s]

100%|██████████| 247/247 [00:01<00:00, 129.26it/s]

100%|██████████| 247/247 [00:01<00:00, 136.23it/s]

100%|██████████| 247/247 [00:01<00:00, 133.66it/s]

100%|██████████| 247/247 [00:01<00:00, 135.86it/s]

100%|██████████| 247/247 [00:01<00:00, 135.87it/s]

100%|██████████| 247/247 [00:01<00:00, 135.68it/s]

100%|██████████| 247/247 [00:01<00:00, 138.33it/s]

100%|██████████| 247/247 [00:01<00:00, 136.39it/s]

100%|██████████| 247/247 [00:01<00:00, 134.14it/s]

100%|██████

In [48]:

new_sino = tomobase.processes.align_tilt_axis_rotation(new_sino)
stackview.slice(new_sino.data)

100%|██████████| 247/247 [00:01<00:00, 132.44it/s]

100%|██████████| 247/247 [00:01<00:00, 133.69it/s]

100%|██████████| 247/247 [00:01<00:00, 133.16it/s]

100%|██████████| 247/247 [00:01<00:00, 132.71it/s]

100%|██████████| 247/247 [00:01<00:00, 136.49it/s]

100%|██████████| 247/247 [00:01<00:00, 133.09it/s]

100%|██████████| 247/247 [00:01<00:00, 135.00it/s]

100%|██████████| 247/247 [00:01<00:00, 132.42it/s]

100%|██████████| 247/247 [00:01<00:00, 132.54it/s]

100%|██████████| 9/9 [00:40<00:00,  4.46s/it]


In [49]:
rec = tomobase.processes.reconstruct.optomo_reconstruct(new_sino, iterations=150)
rec.to_file(os.path.join(directory, subdirectory, 'result_rot90.rec'))
stackview.orthogonal(rec.data)

100%|██████████| 247/247 [04:28<00:00,  1.09s/it]


In [ ]:
#stackview.slice(sino.data)
#normalize sino.data
#sino.data = (sino.data - sino.data.min()) / (sino.data.max() - sino.data.min())
print(np.mean(new_sino.data))


In [ ]:
sino2 = tomobase.processes.align_tilt_axis_rotation(new_sino, inplace=False)
stackview.slice(new_sino.data)